In [ ]:
# One episode, three cold fits, four ensembles; <=20min; no purchases or tokens.
import os,sys,json,time,subprocess,hashlib,platform
from pathlib import Path
started=time.monotonic()
limit=1200
repo=Path('/content/mira_colab_repro')
commit='931534e5d074e1f0a898f4a1c5859a66e0346184'
proof=Path('/content/mira_colab_gpu_proof.json')
def command(argv,cap):
    remaining=limit-(time.monotonic()-started)
    if remaining<=0: raise TimeoutError('20min session bound reached')
    r=subprocess.run(argv,capture_output=True,text=True,timeout=min(cap,remaining))
    if r.returncode: raise RuntimeError(r.stdout[-1500:]+r.stderr[-1500:])
    return r.stdout
# GPU availability must be proven before installing or cloning.
gpu=command(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],30).strip()
print('MIRA_COLAB_GPU_PROOF',gpu,flush=True)
packages=['torch==2.8.0','numpy==2.3.5','scipy==1.17.0','scikit-learn==1.8.0','tabpfn==9.0.0','tabicl==2.2.0']
command([sys.executable,'-m','pip','install','--quiet',*packages],420)
print('MIRA_PINNED_PACKAGES_INSTALLED',flush=True)
if repo.exists(): raise FileExistsError('Preserve existing session directory; stop without retry')
command(['git','clone','--quiet','--filter=blob:none','--no-checkout','--branch','codex/mira-research','https://github.com/saptarshihalder/MIRA.git',str(repo)],120)
command(['git','-C',str(repo),'sparse-checkout','init','--cone'],30)
command(['git','-C',str(repo),'sparse-checkout','set','src','configs','experiments','infra','artifacts/data/natural_compiler'],90)
command(['git','-C',str(repo),'checkout','--quiet',commit],120)
for filename in ['prototype_selector.npz','linear_selector.npz','deepsets_selector.npz']:
    relative='artifacts/reports/matched_compiler_v1/'+filename
    p=repo/relative;p.parent.mkdir(parents=True,exist_ok=True)
    p.write_bytes(subprocess.check_output(['git','-C',str(repo),'show',commit+':'+relative],timeout=30))
assert command(['git','-C',str(repo),'rev-parse','HEAD'],30).strip()==commit
print('MIRA_PINNED_SOURCE',commit,flush=True)
child=r"""
import os,sys,json,time,hashlib,platform,importlib.metadata,gc,warnings
from pathlib import Path
root=Path('/content/mira_colab_repro')
cache=Path('/content/mira_colab_cache');cache.mkdir(exist_ok=True)
os.environ.update(HF_HOME=str(cache/'huggingface'),TABPFN_MODEL_CACHE_DIR=str(cache/'tabpfn'),HF_HUB_DISABLE_IMPLICIT_TOKEN='1',TABPFN_DISABLE_TELEMETRY='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
sys.path.insert(0,str(root/'src'));sys.path.insert(0,str(root/'experiments/mask_compiler'))
plan=json.loads((root/'configs/trained_compiler_validation_v1.json').read_text())
used=['experiments/mask_compiler/run_backbone.py','experiments/mask_compiler/matched_training.py','experiments/mask_compiler/train_selector.py','experiments/mask_compiler/feasibility.py',*['src/mira/'+p.name for p in (root/'src/mira').glob('*.py')],plan['compiler_checkpoint'],plan['linear_checkpoint'],plan['deepsets_checkpoint']]
for relative in used:
    assert hashlib.sha256((root/relative).read_bytes()).hexdigest()==plan['file_sha256'][relative],relative
import numpy as np,torch
from run_backbone import synthetic_episodes,prepare_inputs,load_selectors,select_views
from mira.models import create_model,ModelSettings,predict_binary
assert torch.cuda.is_available(),'GPU required; no CPU substitution'
assert torch.__version__.split('+')[0]=='2.8.0'
versions={p:importlib.metadata.version(p) for p in ['torch','numpy','scipy','scikit-learn','tabpfn','tabicl']}
for p,v in {'numpy':'2.3.5','scipy':'1.17.0','scikit-learn':'1.8.0','tabpfn':'9.0.0','tabicl':'2.2.0'}.items(): assert versions[p]==v
spec={'synthetic':{'orders':[2],'gammas':[.9],'seeds':[74000],'context':128,'queries':256}}
ep=next(synthetic_episodes(spec));support,query=ep['support'],ep['query']
selectors=load_selectors(plan,root);selected=select_views(support,74000,['identity','trained'],selectors)
def array_sha(a):
    return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+np.ascontiguousarray(a).tobytes()).hexdigest()
def nll(p,t):
    p=np.clip(p,1e-6,1-1e-6);return float(np.mean(-t*np.log(p)-(1-t)*np.log1p(-p)))
result={'status':'running','repository_commit':'931534e5d074e1f0a898f4a1c5859a66e0346184','platform':platform.platform(),'python':sys.version.split()[0],'packages':versions,'gpu':torch.cuda.get_device_name(0),'cuda':torch.version.cuda,'ensembles':4,'episode_id':ep['episode_id'],'queries':256,'source_hashes_verified':len(used),'protocol_sha256':hashlib.sha256((root/'configs/trained_compiler_validation_v1.json').read_bytes()).hexdigest(),'data_array_sha256':{k:array_sha(a) for k,a in {'X_support':ep['native_context'],'X_query':ep['native_query'],'u_support':support.observations.u,'u_query':query.observations.u,'y_support':support.labels,'y_query':ep['targets'],'oracle':ep['oracle']}.items()},'selection':selected,'predictions':{},'warnings':[],'additional_purchase':False}
begun=time.monotonic()
for name,mode in [('identity','identity'),('trained','trained'),('trained_repeat','trained')]:
    inputs=prepare_inputs(support,query,ep['selected_columns'],np.array(selected[mode]['matrix']),mode,74000,ep['native_context'],ep['native_query'])
    assert np.array_equal(inputs.context[:,1:5],ep['native_context'].astype(np.float32),equal_nan=True)
    assert np.array_equal(inputs.query[:,1:5],ep['native_query'].astype(np.float32),equal_nan=True)
    with warnings.catch_warnings(record=True) as ws:
        model=create_model('tabpfn:v2',ModelSettings('cuda',4),74000)
        probability=predict_binary(model,inputs)
    result['warnings'].extend(str(w.message)[:300] for w in ws)
    result['predictions'][name]={'probability':probability.tolist(),'probability_array_sha256':array_sha(probability),'expected_nll':nll(probability,ep['oracle']),'empirical_nll':nll(probability,ep['targets']),'input_shapes':[list(inputs.context.shape),list(inputs.query.shape)]}
    del model;gc.collect();torch.cuda.empty_cache()
    print('MIRA_FIT_COMPLETE',name,flush=True)
checkpoints=list((cache/'tabpfn').rglob('tabpfn-v2-classifier-finetuned-zk73skhh.ckpt'))
assert len(checkpoints)==1,'Exact v2 checkpoint must be found once'
digest=hashlib.sha256(checkpoints[0].read_bytes()).hexdigest()
assert digest==plan['checkpoint_sha256']['tabpfn/tabpfn-v2-classifier-finetuned-zk73skhh.ckpt'],'Checkpoint SHA differs'
a=np.array(result['predictions']['trained']['probability']);b=np.array(result['predictions']['trained_repeat']['probability'])
result.update(status='complete',native_arrays_preserved=True,checkpoint_sha256=digest,repeat_max_abs_difference=float(np.max(np.abs(a-b))),repeat_exact_equal=bool(np.array_equal(a,b)),seconds=time.monotonic()-begun)
Path('/content/mira_colab_gpu_proof.json').write_text(json.dumps(result,allow_nan=False))
print('MIRA_REPRODUCTION_COMPLETE',json.dumps({k:result[k] for k in ['status','gpu','checkpoint_sha256','repeat_max_abs_difference','seconds']}),flush=True)
"""
script=repo/'colab_repro.py';script.write_text(child)
print(command([sys.executable,str(script)],max(1,min(600,limit-(time.monotonic()-started)))),flush=True)
from IPython.display import display,HTML
import base64
blob=base64.b64encode(proof.read_bytes()).decode()
display(HTML('<a download="mira_colab_gpu_proof.json" href="data:application/json;base64,'+blob+'">Download MIRA GPU proof</a>'))
print('MIRA_TOTAL_SECONDS',time.monotonic()-started,flush=True)
